# Relatório Pré-Decolagem

Este notebook unifica as etapas de validação de telemetria, análise energética e assistência por inteligência artificial para a decisão final de lançamento.

---
## 1.1 Organização e descrição da telemetria

A telemetria da espaçonave é monitorada com base nos seguintes parâmetros e suas respectivas faixas de segurança operacionais para decolagem:

- **Temperatura Interna**: entre 18°C e 24°C
- **Temperatura Externa**: entre -150°C e 120°C
- **Integridade Estrutural**: Obrigatório 1 (100% OK)
- **Nível de Energia**: Mínimo de 80%
- **Nível de Combustível**: Mínimo de 90%
- **Pressão do Tanque de Combustível**: entre 200 atm e 300 atm
- **Pressão do Tanque Oxidante**: entre 200 atm e 300 atm
- **Status dos Módulos Críticos**: Obrigatoriamente 'S' (Sem erros)

---
## 1.2 Algoritmo de verificação

Abaixo apresentamos o fluxograma do algoritmo responsável por decidir se o status é **PRONTO PARA DECOLAR** ou **DECOLAGEM ABORTADA**, avaliando cada campo da telemetria contra os limites estabelecidos.

![Fluxograma de Verificação](fluxograma.png)

---
## 1.3 Script em Python

Abaixo está a implementação lógica do algoritmo de telemetria ilustrado acima.

In [2]:
# Dicionário com faixas de segurança definidas
LIMITES_MISSAO = {
    "temp_int_min": 18,
    "temp_int_max": 24,
    "temp_ext_min": -150,
    "temp_ext_max": 120, # graus celsius
    "integridade_requerida": 1,
    "energia_min": 80, # porcentagem
    "nivel_combustivel_min": 90,   # porcentagem
    "pressao_tanque_combust_min": 200, # atm
    "pressao_tanque_combust_max": 300, # atm
    "pressao_tanque_oxidante_min": 200,
    "pressao_tanque_oxidante_max": 300,
    "status_modulos_esperado": "S"
}

def input_telemetria():
    print("---> ENTRADA DE DADOS DE PRÉ-DECOLAGEM <---")

    telemetria = {
        "temp_interna": float(input("Temperatura Interna (°C): ")),
        "temp_externa":  float(input("Temperatura Externa (°C): ")),
        "integridade": int(input("Integridade Estrutural (1-OK / 0-Falha): ")),
        "energia_pct": float(input("Nível de Energia (%): ")),
        "combustivel_pct": float(input("Nível de Combustível (%): ")),
        "pressao_tanque_combust_atm": float(input("Pressão do Tanque de Combustível (atm): ")),
        "pressao_tanque_oxidante_atm": float(input("Pressão do Tanque Oxidante (atm): ")),
        "status_modulos": input("Módulos Críticos estão OK? (S/N): ").upper()
    }
    return telemetria

def verificar_sistemas(dados, limites):
    erros = []

    if not (limites["temp_int_min"] <= dados["temp_interna"] <= limites["temp_int_max"]):
        erros.append(f"Temperatura Interna fora da faixa (entre {limites['temp_int_min']}°C e {limites['temp_int_max']}°C)")
    if not (limites["temp_ext_min"] <= dados["temp_externa"] <= limites["temp_ext_max"]):
        erros.append(f"Temperatura Externa fora da faixa (entre {limites['temp_ext_min']}°C e {limites['temp_ext_max']}°C)")
    if dados["integridade"] != limites["integridade_requerida"]:
        erros.append("Falha na Integridade Estrutural")
    if dados["energia_pct"] < limites["energia_min"]:
        erros.append(f"Nível de energia abaixo do mínimo esperado: ({limites['energia_min']}%)")
    if dados["combustivel_pct"] < limites["nivel_combustivel_min"]:
        erros.append(f"Nível de combustível abaixo do mínimo esperado: ({limites['nivel_combustivel_min']}%)")
    if not (limites["pressao_tanque_combust_min"] <= dados["pressao_tanque_combust_atm"] <= limites["pressao_tanque_combust_max"]):
        erros.append(f"Pressão do Tanque de Combustível fora da faixa (entre {limites['pressao_tanque_combust_min']} atm e {limites['pressao_tanque_combust_max']} atm)")
    if not (limites["pressao_tanque_oxidante_min"] <= dados["pressao_tanque_oxidante_atm"] <= limites["pressao_tanque_oxidante_max"]):
        erros.append(f"Pressão do Tanque Oxidante fora da faixa (entre {limites['pressao_tanque_oxidante_min']} atm e {limites['pressao_tanque_oxidante_max']} atm)")
    if dados["status_modulos"] != limites["status_modulos_esperado"]:
        erros.append("Módulos Críticos com erros")

    if len(erros) == 0:
        return True, "PRONTO PARA DECOLAGEM!"
    else:
        return False, f"DECOLAGEM ABORTADA! Motivos: {', '.join(erros)}"

dados_telemetria = input_telemetria()
sucesso, mensagem = verificar_sistemas(dados_telemetria, LIMITES_MISSAO)

print("\n" + "="*25)
print(mensagem)
print("="*25)

---> ENTRADA DE DADOS DE PRÉ-DECOLAGEM <---
Temperatura Interna (°C): 15
Temperatura Externa (°C): 125
Integridade Estrutural (1-OK / 0-Falha): 0
Nível de Energia (%): 20
Nível de Combustível (%): 65
Pressão do Tanque de Combustível (atm): 5
Pressão do Tanque Oxidante (atm): 98
Módulos Críticos estão OK? (S/N): n

DECOLAGEM ABORTADA! Motivos: Temperatura Interna fora da faixa (entre 18°C e 24°C), Temperatura Externa fora da faixa (entre -150°C e 120°C), Falha na Integridade Estrutural, Nível de energia abaixo do mínimo esperado: (80%), Nível de combustível abaixo do mínimo esperado: (90%), Pressão do Tanque de Combustível fora da faixa (entre 200 atm e 300 atm), Pressão do Tanque Oxidante fora da faixa (entre 200 atm e 300 atm), Módulos Críticos com erros


---
## 1.4 Análise energética

Nesta etapa calculamos a autonomia inicial projetada. Consideramos a **Capacidade total (kWh)**, a **Carga atual (%)** alimentada pela telemetria, bem como o **Consumo estimado na decolagem** e **Perdas energéticas** predefinidas.

In [ ]:
# Definindo as variáveis e constantes operacionais 
CAPACIDADE_TOTAL_KWH = 50000.0  # Capacidade máxima de kWh das baterias do foguete
CONSUMO_DECOLAGEM_KWH = 30000.0  # Consumo previsto (ignição + ascensão principal)
TAXA_PERDA_PCT = 2.0  # Perda estimada em conversão e sistemas térmicos (%)

In [ ]:
def calcular_autonomia(energia_pct):
    """
    Calcula o saldo de energia após a decolagem simulada e emite um veredito de aprovação.
    """
    # Capacidade real com base no nível atual (%) da bateria
    energia_inicial_kwh = CAPACIDADE_TOTAL_KWH * (energia_pct / 100)
    
    # Cálculo das perdas inerentes e disponibilidade bruta e líquida
    perdas_kwh = energia_inicial_kwh * (TAXA_PERDA_PCT / 100)
    energia_liquida_disponivel = energia_inicial_kwh - perdas_kwh
    
    # Saldo final projetado após o consumo da decolagem
    saldo_energia_pos_decolagem = energia_liquida_disponivel - CONSUMO_DECOLAGEM_KWH
    
    print("\n=== ANÁLISE ENERGÉTICA DE PRÉ-DECOLAGEM ===")
    print(f"Capacidade Total: {CAPACIDADE_TOTAL_KWH:.2f} kWh")
    print(f"Carga Atual das Baterias: {energia_pct}%")
    print(f"-> Energia Inicial Bruta: {energia_inicial_kwh:.2f} kWh")
    print(f"-> Perdas Estimadas ({TAXA_PERDA_PCT}%): {perdas_kwh:.2f} kWh")
    print(f"-> Energia Líquida para Voo: {energia_liquida_disponivel:.2f} kWh")
    print(f"-> Consumo da Decolagem: {CONSUMO_DECOLAGEM_KWH:.2f} kWh")
    
    print("\n--- DIAGNÓSTICO DE AUTONOMIA ---")
    if saldo_energia_pos_decolagem > 0:
        print("[Veredito]: ✅ PRONTO PARA DECOLAR")
        print(f"[Saldo Estimado]: {saldo_energia_pos_decolagem:.2f} kWh restando após a inserção na órbita.")
        print("Conclusão: Capacidade energética é suficiente para a manobra inicial e contingência.")
        return True
    else:
        print("[Veredito]: ❌ DECOLAGEM ABORTADA")
        print(f"[Déficit Previsto]: {abs(saldo_energia_pos_decolagem):.2f} kWh.")
        print("Conclusão: Autonomia INSUFICIENTE. A espaçonave ficaria sem energia durante o voo.")
        return False

In [ ]:
# Execução do script solicitando a leitura de telemetria
try:
    carga_atual = float(input("Informe o nível de carga atual (%) fornecido pela telemetria: "))
    if 0 <= carga_atual <= 100:
        calcular_autonomia(carga_atual)
    else:
        print("Erro: A carga das baterias deve ser um valor entre 0 e 100.")
except ValueError:
    print("Entrada inválida. Por favor, insira um número para a porcentagem de bateria.")

---
## 1.5 Análise assistida por IA

Solicitamos a uma Inteligência Artificial para validar a **classificação dos dados**, apontar possíveis **anomalias** que o sistema linear possa não reparar e sugerir avaliações de **risco** para a decolagem.

In [ ]:
import requests
import json
from google.colab import userdata
from IPython.display import Markdown, display

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
MODELO = "stepfun/step-3.5-flash:free"
OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

telemetria = {
    "Temperatura Interna": {"valor": 22.5, "unidade": "°C", "faixa_segura": "-10 a 50°C"},
    "Temperatura Externa": {"valor": -120.0, "unidade": "°C", "faixa_segura": "-200 a -50°C"},
    "Integridade Estrutural": {"valor": 92, "unidade": "%", "faixa_segura": "≥ 90%"},
    "Nível de Energia (Bateria)": {"valor": 78, "unidade": "%", "faixa_segura": "≥ 70%"},
    "Nível de Combustível": {"valor": 95, "unidade": "%", "faixa_segura": "≥ 80%"},
    "Pressão Tanque A": {"valor": 1.8, "unidade": "bar", "faixa_segura": "1.5 a 2.5 bar"},
    "Pressão Tanque B": {"valor": 2.1, "unidade": "bar", "faixa_segura": "1.5 a 2.5 bar"},
    "Status Módulos Críticos": {"valor": "Todos OK", "unidade": "-", "faixa_segura": "OK"},
}

linhas = []
for param, info in telemetria.items():
    linhas.append(
        f"- {param}: {info['valor']} {info['unidade']} (faixa segura: {info['faixa_segura']})"
    )
dados_formatados = "\n".join(linhas)
print("Dados de telemetria preparados:\n")
print(dados_formatados)

prompt = f"""Você é um especialista em sistemas espaciais e análise de telemetria.
Analise os dados abaixo de uma nave em pré-decolagem e forneça:

1. **Classificação dos dados**: Para cada parâmetro, classifique como NORMAL, ATENÇÃO ou CRÍTICO.
2. **Anomalias detectadas**: Liste quaisquer valores fora do padrão ou que mereçam atenção.
3. **Sugestões de risco**: Aponte os principais riscos para a missão e recomendações.
4. **Parecer final**: A nave está apta ou não para decolagem segundo esta análise?

Dados de telemetria:
{dados_formatados}

Responda em português, de forma técnica e objetiva.
"""

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": MODELO,
    "messages": [
        {"role": "user", "content": prompt}
    ]
}

print("Enviando dados para análise pela IA...\n")
response = requests.post(OPENROUTER_API_URL, headers=headers, json=payload)
data = response.json()

if "choices" in data:
    modelo_usado = data.get("model", MODELO)
    resposta_ia = data["choices"][0]["message"]["content"]
    display(Markdown("---\n## 🤖 Análise Assistida por IA\n\n" + resposta_ia))
else:
    print(json.dumps(data, indent=2))


Dados de telemetria preparados:

- Temperatura Interna: 22.5 °C (faixa segura: -10 a 50°C)
- Temperatura Externa: -120.0 °C (faixa segura: -200 a -50°C)
- Integridade Estrutural: 92 % (faixa segura: ≥ 90%)
- Nível de Energia (Bateria): 78 % (faixa segura: ≥ 70%)
- Nível de Combustível: 95 % (faixa segura: ≥ 80%)
- Pressão Tanque A: 1.8 bar (faixa segura: 1.5 a 2.5 bar)
- Pressão Tanque B: 2.1 bar (faixa segura: 1.5 a 2.5 bar)
- Status Módulos Críticos: Todos OK - (faixa segura: OK)
Enviando dados para análise pela IA...



---
## 🤖 Análise Assistida por IA

### 1. Classificação dos Dados
| Parâmetro               | Valor       | Faixa Segura          | Classificação |
|-------------------------|-------------|-----------------------|---------------|
| Temperatura Interna     | 22.5 °C     | -10 a 50 °C           | NORMAL        |
| Temperatura Externa     | -120.0 °C   | -200 a -50 °C         | NORMAL        |
| Integridade Estrutural  | 92 %        | ≥ 90 %                | NORMAL*       |
| Nível de Energia        | 78 %        | ≥ 70 %                | NORMAL        |
| Nível de Combustível    | 95 %        | ≥ 80 %                | NORMAL        |
| Pressão Tanque A        | 1.8 bar     | 1.5 a 2.5 bar         | NORMAL        |
| Pressão Tanque B        | 2.1 bar     | 1.5 a 2.5 bar         | NORMAL        |
| Status Módulos Críticos | Todos OK    | OK                    | NORMAL        |

*Observação: A Integridade Estrutural está no limite inferior aceitável (92% ≥ 90%), mas dentro da faixa.

---

### 2. Anomalias Detectadas
- **Nenhum parâmetro está fora da faixa segura definida.** Todos os valores atendem aos critérios mínimos de segurança.
- Observação crítica: **Integridade Estrutural em 92%** representa margem operacional reduzida, mas não caracteriza falha.

---

### 3. Sugestões de Risco e Recomendações
**Riscos Principais:**
- **Margem reduzida na Integridade Estrutural (92%)**: Valores próximos ao limite mínimo aumentam a vulnerabilidade a cargas dinâmicas (vibrações, pressão aerodinâmica) durante a decolagem e fase de Max Q.
- Possível degradação não监测ada: Se houver tendência de queda na integridade estrutural em missões anteriores, Flight Safety pode ser comprometido.

**Recomendações Imediatas:**
1. **Monitoramento contínuo** da Integridade Estrutural durante todas as fases de voo, com atenção especial à decolagem e transição supersônica.
2. **Verificação de fontes de estresse** (vibrações de motores, carga de manobras) que possam acelerar a degradação estrutural.
3. **Revisão de dados históricos** da integridade estrutural para identificar tendências de decaimento.
4. **Inspeção pós-voo** detalhada da fuselagem e estruturas críticas, independentemente do resultado da missão.

---

### 4. Parecer Final
**A nave está APTA para decolagem**, pois todos os parâmetros de telemetria estão dentro das faixas seguras estabelecidas. No entanto, a **Integridade Estrutural no limite inferior (92%) exige vigilância operacional reforçada** durante a missão. Recomenda-se priorizar o monitoramento desse parâmetro e avaliar a necessidade de margin of safety adicional em futuras missões.

**Decisão:** DECOLAGEM AUTORIZADA com mitigação de risco ativa para Integridade Estrutural.